# B07-02 — 배지 조성으로 성장 특성 예측: Ridge와 Random Forest

**BioCulture · 실제 데이터 · 학습용 Notebook v1**

## 학습 목표
1. B07-00의 **44개 성분 농도**와 B07-01의 **배지 조성별 목표 중앙값**을 안전하게 결합한다.
2. 고유 배지 조성(`medium_profile_id`)을 독립적인 분석 단위로 정의한다.
3. 훈련·시험 분할 → 훈련 내부 교차 검증 → 최종 시험 평가 순서로 Ridge와 Random Forest를 비교한다.
4. 반복 성장곡선과 배지 조성의 혼동, 실험 라운드 교란, 데이터 누출, 관찰 상관관계의 한계를 기록한다.

> **주의:** 이 Notebook은 배지 성분의 인과 효과나 미실험 배지에서의 실제 효능을 입증하지 않습니다. 원본 `K`, `r`의 계산 정의 역시 별도로 확인해야 합니다. 여기서는 원본 제공 성장 지표를 **조성별 중앙값으로 요약한 예측 목표**로만 사용합니다.

**실행 환경:** Windows/Anaconda `bioimage-ai`, Python 3.11, `numpy`, `pandas`, `scikit-learn`, `matplotlib`. 이전 B07-00 Clean v2, B07-01 실행 결과가 필요합니다.

## 1. 프로젝트·파일 경로와 선행 감사 상태

In [ ]:
from pathlib import Path
import hashlib, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

cwd=Path.cwd().resolve()
ROOT=next((p for p in [cwd,cwd.parent,cwd/'bioimage-ai'] if (p/'results'/'bioculture'/'B07_00_ml_data_audit_v2').is_dir()),None)
if ROOT is None: raise FileNotFoundError('B07_00_ml_data_audit_v2 폴더를 찾을 수 없습니다. 프로젝트 루트에서 실행하세요.')
R=ROOT/'results'/'bioculture'; A=R/'B07_00_ml_data_audit_v2'; E=R/'B07_01_medium_growth_eda_v1'; OUT=R/'B07_02_medium_growth_ml_v1'; OUT.mkdir(parents=True,exist_ok=True)
source_profile=A/'ml_medium_profile_level_audit_table.csv'
source_feature=A/'medium_feature_audit.csv'
source_eda=E/'eda_medium_profile_table.csv'
source_curve=A/'ml_curve_level_audit_table.csv'
source_report=A/'B07_00_validation_report.json'
print('Project:',ROOT)
print('Python:',sys.version.split()[0])
print('scikit-learn:',__import__('sklearn').__version__)
print('Output:',OUT)
print('B07-00 reports:',[p.name for p in A.glob('*.json')])

## 2. 감사 통과 확인 및 입력 데이터 구조 재검증

열 이름을 별도의 수동 목록으로 재입력하지 않습니다. B07-00의 성분 감사 파일에서 44개 입력 성분을 읽습니다. 식별자·목표·라운드·품질 플래그가 성분 변수에 포함되지 않았는지 검증합니다.

In [ ]:
for p in (source_profile,source_feature,source_eda,source_curve):
    if not p.is_file(): raise FileNotFoundError(p)
reports=list(A.glob('*validation*report*.json'))
if reports:
    prior=json.loads(reports[0].read_text(encoding='utf-8'))
    print('B07-00 status:',prior.get('status'))
    assert prior.get('status')=='COMPOSITION_AUDIT_PASSED_NOT_MODEL_VALIDATION'
profiles=pd.read_csv(source_profile,encoding='utf-8-sig',dtype={'medium_profile_id':'string'},low_memory=False)
eda=pd.read_csv(source_eda,encoding='utf-8-sig',dtype={'medium_profile_id':'string'},low_memory=False)
fa=pd.read_csv(source_feature,encoding='utf-8-sig'); features=fa['feature'].astype(str).tolist()
assert len(features)==44 and len(set(features))==44
assert len(profiles)==len(eda)==1025
assert profiles.medium_profile_id.is_unique and eda.medium_profile_id.is_unique
assert set(profiles.medium_profile_id)==set(eda.medium_profile_id)
assert set(features).issubset(profiles.columns)
assert set(['K_median','r_median','n_curves','n_rounds']).issubset(eda.columns)
ban={'Label','Assay ID','Condition ID','medium_profile_id','round','K','r','K_median','r_median'}
assert not ban.intersection(features) and not any(str(x).startswith('flag_') for x in features)
for f in features:
    assert pd.to_numeric(profiles[f],errors='coerce').notna().all(), f
Xall=profiles.set_index('medium_profile_id')[features].astype(float).sort_index()
met=eda.set_index('medium_profile_id').reindex(Xall.index)
assert len(Xall)==1025 and not Xall.index.has_duplicates
curve_min=pd.read_csv(source_curve,encoding='utf-8-sig',usecols=['medium_profile_id'],dtype={'medium_profile_id':'string'})
assert set(curve_min.medium_profile_id)==set(Xall.index)
assert len(curve_min)==13608
print('Profiles',len(Xall),'Features',Xall.shape[1],'Curves (metadata check)',len(curve_min))
print('Target availability:',{k:int(met[k].notna().sum()) for k in ['K_median','r_median']})

## 3. 시험 집합 먼저 고정하기 — 모든 목표에 동일한 조성 분할

학습 데이터에만 기반하여 결측 대체·표준화·하이퍼파라미터 선택을 수행합니다. 같은 조성의 반복 곡선은 분할에 관여하지 않습니다. `medium_profile_id` 기준으로 조성이 서로 겹치지 않는지 확인합니다. **목표값을 보고 시험 조성을 선택하지 않습니다.**

In [ ]:
SEED=20261019
ids=np.asarray(Xall.index.astype(str))
ids_train,ids_test=train_test_split(ids,test_size=0.20,random_state=SEED,shuffle=True)
assert set(ids_train).isdisjoint(set(ids_test))
assert len(ids_train)+len(ids_test)==len(Xall)
splits=pd.DataFrame({'medium_profile_id':ids,'split':['train' if x in set(ids_train) else 'test' for x in ids]})
splits.to_csv(OUT/'medium_profile_train_test_split.csv',index=False,encoding='utf-8-sig')
# 곡선 수준 데이터로도 누출 없는지 검사: 동일 조성의 전체 반복이 한쪽에만 속함
curve_splits=curve_min.merge(splits,on='medium_profile_id',validate='many_to_one')
assert len(curve_splits)==len(curve_min)
assert (curve_splits.groupby('medium_profile_id').split.nunique()==1).all()
print('Train profiles:',len(ids_train),'Test profiles:',len(ids_test))
print('Curve counts in corresponding groups:',curve_splits.split.value_counts().to_dict())

## 4. 모델·평가 절차 정의

`DummyRegressor`는 비교 기준입니다. Ridge는 학습 자료에서만 `StandardScaler`를 학습하고, Random Forest는 스케일 조정 없이 사용합니다. 두 모델의 하이퍼파라미터는 **훈련 자료 내부 5-fold CV**에서만 고릅니다. 독립 변수 44개 외 메타데이터는 입력하지 않습니다.

평가 지표는 MAE·RMSE·R²입니다. R²가 음수일 수 있으며, 이는 시험 자료에서 평균 예측보다 나쁠 수 있음을 뜻합니다.

In [ ]:
def build_estimators(seed=SEED):
    ridge=Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',Ridge())])
    forest=Pipeline([('imputer',SimpleImputer(strategy='median')),('model',RandomForestRegressor(n_estimators=160,random_state=seed,n_jobs=-1,min_samples_leaf=2))])
    dummy=Pipeline([('imputer',SimpleImputer(strategy='median')),('model',DummyRegressor(strategy='median'))])
    return {'DummyMedian':(dummy,{}), 'Ridge':(ridge,{'model__alpha':[0.1,1,10,100,1000]}),
            'RandomForest':(forest,{'model__max_features':[0.5,1.0],'model__min_samples_leaf':[2,5]})}

def metrics(y,yp):
    return {'MAE':float(mean_absolute_error(y,yp)),
            'RMSE':float(np.sqrt(mean_squared_error(y,yp))),
            'R2':float(r2_score(y,yp))}

def run_target(target):
    y=pd.to_numeric(met[target],errors='coerce')
    train_valid=[x for x in ids_train if pd.notna(y.loc[x])]
    test_valid=[x for x in ids_test if pd.notna(y.loc[x])]
    assert len(train_valid)>=100 and len(test_valid)>=20
    Xtr=Xall.loc[train_valid]; Xte=Xall.loc[test_valid]
    ytr=y.loc[train_valid].astype(float); yte=y.loc[test_valid].astype(float)
    cv=KFold(n_splits=5,shuffle=True,random_state=SEED)
    rows=[]; preds=[]; cvrows=[]; fitted={}
    for model_name,(est,grid) in build_estimators().items():
        if grid:
            search=GridSearchCV(est,grid,scoring='neg_mean_absolute_error',cv=cv,n_jobs=1,refit=True,return_train_score=False)
            search.fit(Xtr,ytr)
            fit=search.best_estimator_
            best=search.best_params_
            cv_mae=-float(search.best_score_)
            for j,sc in enumerate(search.cv_results_['mean_test_score']):
                cvrows.append({'target':target,'model':model_name,'params':str(search.cv_results_['params'][j]),'cv_MAE':-float(sc)})
        else:
            fit=est.fit(Xtr,ytr);best={};cv_mae=float('nan')
        yp=fit.predict(Xte)
        row={'target':target,'model':model_name,'n_train_profiles':len(train_valid),
             'n_test_profiles':len(test_valid),'best_params':json.dumps(best,ensure_ascii=False),
             'train_CV_MAE':cv_mae,**metrics(yte,yp)}
        rows.append(row)
        for mid,obs,pred in zip(test_valid,yte,yp):
            preds.append({'target':target,'model':model_name,'medium_profile_id':mid,'y_true':float(obs),'y_pred':float(pred),'residual':float(obs-pred)})
        fitted[model_name]=fit
        print(target,model_name,'MAE',round(row['MAE'],5),'R2',round(row['R2'],4))
    return pd.DataFrame(rows),pd.DataFrame(preds),pd.DataFrame(cvrows),fitted

## 5. 실제 모델 학습 및 독립 시험 집합 평가

기본 목표는 논문의 원본 평가값 `K`, `r`의 조성별 중앙값입니다. 학습은 **1025개 조성 중 목표가 존재하는 조성만** 사용하며, 결측 목표를 평균값으로 채우지 않습니다. 이 결과는 예측에 대한 탐색적 검증이지 배지 성분의 실험적 효과 검증이 아닙니다.

In [ ]:
result_parts=[];pred_parts=[];cv_parts=[];fitted_models={}
for target in ('K_median','r_median'):
    result,preds,cv_results,fitted=run_target(target)
    result_parts.append(result);pred_parts.append(preds);cv_parts.append(cv_results)
    fitted_models[target]=fitted
results=pd.concat(result_parts,ignore_index=True)
predictions=pd.concat(pred_parts,ignore_index=True)
cv_all=pd.concat(cv_parts,ignore_index=True)
assert len(results)==6
assert predictions.groupby(['target','model']).medium_profile_id.nunique().min()>=20
results.to_csv(OUT/'holdout_model_comparison.csv',index=False,encoding='utf-8-sig')
predictions.to_csv(OUT/'holdout_predictions.csv',index=False,encoding='utf-8-sig')
cv_all.to_csv(OUT/'train_only_cv_candidates.csv',index=False,encoding='utf-8-sig')
print(results.to_string(index=False))

## 6. 예측값 시각화와 잔차 감사

검증용 산점도는 이미 고정된 시험 조성만 사용합니다. 시험 결과를 본 뒤 모델이나 매개변수를 다시 조정하여 같은 시험 집합을 재사용하면 평가가 낙관적으로 변할 수 있습니다.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4.3))
for ax,target in zip(axes,['K_median','r_median']):
    sub=predictions[predictions.target==target]
    for model in ['DummyMedian','Ridge','RandomForest']:
        x=sub[sub.model==model]
        ax.scatter(x.y_true,x.y_pred,s=13,alpha=.5,label=model)
    lo=min(sub.y_true.min(),sub.y_pred.min()); hi=max(sub.y_true.max(),sub.y_pred.max())
    ax.plot([lo,hi],[lo,hi],ls='--',lw=1,color='gray')
    ax.set(xlabel='Observed (profile median)',ylabel='Predicted',title=target)
    ax.legend(fontsize=8)
fig.tight_layout();fig.savefig(OUT/'holdout_observed_vs_predicted.png',dpi=160)
plt.show()
print(predictions.groupby(['target','model']).residual.agg(['count','median','std']).round(4))

## 7. 라운드 교란 가능성 감사

이 실험에서 라운드 간 측정 수준의 차이가 큽니다. `round`를 예측 변수에 넣지 않더라도 **배지 조성이 특정 라운드에 치우쳐 있다면 라운드 차이를 간접적으로 학습할 수 있습니다.** 이번 단계에서는 라운드별 시험 오차를 *진단*하며, 완전히 새로운 라운드에 대한 외삽 성능은 입증하지 않습니다.

In [ ]:
cmeta=pd.read_csv(source_curve,encoding='utf-8-sig',usecols=['medium_profile_id','round'],dtype={'medium_profile_id':'string'})
rounds=cmeta.groupby('medium_profile_id')['round'].agg(lambda s:','.join(sorted(set(str(x) for x in s.dropna())))).rename('round_membership')
held=predictions.merge(rounds,left_on='medium_profile_id',right_index=True,validate='many_to_one')
round_audit=held.groupby(['target','model','round_membership']).agg(n_test_profiles=('medium_profile_id','nunique'),MAE=('residual',lambda s:float(np.abs(s).mean()))).reset_index()
round_audit.to_csv(OUT/'holdout_round_membership_audit.csv',index=False,encoding='utf-8-sig')
print('Held-out profiles per round membership; multiple-round profiles are separate membership strings:')
print(round_audit[round_audit.model=='Ridge'].to_string(index=False,max_rows=25))
print('Distinct multiround profiles:',int((rounds.str.contains(',')).sum()))

## 8. 저장 결과·재현성·무결성 확인

In [ ]:
reloaded=pd.read_csv(OUT/'holdout_model_comparison.csv',encoding='utf-8-sig')
assert len(reloaded)==6
assert set(reloaded.model)=={'DummyMedian','Ridge','RandomForest'}
assert set(reloaded.target)=={'K_median','r_median'}
assert set(pd.read_csv(OUT/'medium_profile_train_test_split.csv',encoding='utf-8-sig').split)=={'train','test'}
assert set(ids_train).isdisjoint(set(ids_test))

def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()
manifest=pd.DataFrame([{'kind':kind,'file':str(p.relative_to(ROOT)),'sha256':sha256(p),'bytes':p.stat().st_size}
    for kind,paths in [('input',[source_profile,source_feature,source_eda,source_curve]),('output',sorted(x for x in OUT.iterdir() if x.is_file() and x.suffix in ('.csv','.png')))]
    for p in paths])
manifest.to_csv(OUT/'file_manifest_sha256.csv',index=False,encoding='utf-8-sig')
report={'status':'HOLDOUT_EVALUATED_EXPLORATORY_NOT_CAUSAL', 'n_profiles':int(len(Xall)),
        'n_features':len(features),'split_seed':SEED,'test_fraction':.2,
        'n_train_profiles':len(ids_train),'n_test_profiles':len(ids_test),
        'n_curves_for_group_audit':int(len(curve_min)),
        'train_test_profile_overlap':int(len(set(ids_train)&set(ids_test))),
        'targets':['K_median','r_median'], 'n_models':3,
        'warning':'No independent round-extrapolation validation; observational associations, not causal effects.',
        'holdout_metrics':results.to_dict(orient='records')}
(OUT/'B07_02_validation_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('Report status:',report['status'])
print('Files:',[x.name for x in sorted(OUT.iterdir())])

## 9. 결과 해석과 과학적 한계

- **새로운 배지 조성 일반화:** 이번 시험은 학습에서 보지 않은 *조성*으로의 내부 일반화를 평가한다. 그러나 모든 실험 라운드에 대한 일반화가 보장되는 것은 아니다.
- **반복 측정:** 개별 성장곡선이 아니라 배지 조성별 중앙값을 하나의 사례로 사용한다. 반복 수의 불균형과 측정 불확실성은 추후 별도로 모델링해야 한다.
- **목표 변수:** `K_median`과 `r_median`은 원본 보고서의 `K`, `r`을 조성별로 요약한 지표다. 생물학적 정의·측정 보정 상태는 이 Notebook에서 새로 검증하지 않는다.
- **설명과 인과:** Ridge 계수, Random Forest 중요도가 높더라도 해당 성분의 인과적 효과라고 주장할 수 없다.
- **향후 B07-03:** 그룹 기반 반복 검증, 조성-라운드 교란, 다른 분할 시드에서 결과 변동을 확인한다.

## 연습문제 — 정답과 Python 코드는 별도의 Word 해설서에 수록

1. 같은 `medium_profile_id`의 모든 성장곡선이 동일한 분할에 들어가는 이유를 설명하고, 이를 확인하는 검증 코드를 작성하라.
2. Ridge와 Random Forest의 전처리 파이프라인이 왜 다른지 설명하고, 표준화를 시험 자료에서 별도로 학습하면 어떤 문제가 생기는지 보이라.
3. `K_median` 예측에서 MAE와 R²가 서로 다른 결론을 주는 상황을 설명하고 각 모델을 기준 모델과 비교하라.
4. 시험 조성별 잔차가 어떤 라운드에서 크게 나타나는지 조사하고, 이를 단일 배지 성분 효과로 해석할 수 없는 이유를 기술하라.
5. `r_median` 목표의 유효 조성 수가 `K_median`과 다른 이유를 확인하고 결측 목표값을 0으로 채우면 안 되는 이유를 설명하라.
